In [4]:
#On charge le fichier en lecture partielle pour ne pas saturer la RAM
import pandas as pd

# Chargement du dataset (on peut limiter à une partie pour tester)
df = pd.read_csv("dvf.csv.gz", compression="gzip", low_memory=False, nrows=200_000)

# Aperçu rapide
df.head(3)
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))
df.columns
colonnes_utiles = [
    "date_mutation", "valeur_fonciere", "code_postal", "nom_commune",
    "type_local", "surface_reelle_bati", "nombre_pieces_principales",
    "longitude", "latitude"
]
df = df[colonnes_utiles].copy()
df.head()


Nombre de lignes : 200000
Nombre de colonnes : 40


,date_mutation,valeur_fonciere,code_postal,nom_commune,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude
0,2020-01-07,8000.0,1250.0,Ceyzériat,NaN,NaN,NaN,5.323532,46.171941
1,2020-01-02,2175.0,1290.0,Laiz,NaN,NaN,NaN,4.893454,46.251858
2,2020-01-02,2175.0,1290.0,Laiz,NaN,NaN,NaN,4.900210,46.235277
3,2020-01-02,2175.0,1290.0,Laiz,NaN,NaN,NaN,4.882112,46.246554
4,2020-01-02,2175.0,1290.0,Laiz,NaN,NaN,NaN,4.894481,46.251841


In [9]:
# Supprimer lignes sans valeur foncière ou surface
df = df.dropna(subset=["valeur_fonciere", "surface_reelle_bati", "type_local"])

# Convertir les colonnes numériques
df["valeur_fonciere"] = pd.to_numeric(df["valeur_fonciere"], errors="coerce")
df["surface_reelle_bati"] = pd.to_numeric(df["surface_reelle_bati"], errors="coerce")
df["nombre_pieces_principales"] = pd.to_numeric(df["nombre_pieces_principales"], errors="coerce")
#filter par surface et prix
df["prix_m2"] = df["valeur_fonciere"] / df["surface_reelle_bati"]

#Supprimer les ventes absurdes (prix très bas ou très haut).
df = df[(df["prix_m2"] > 500) & (df["prix_m2"] < 10000)]
df = df[df["surface_reelle_bati"] > 10]  # exclure caves, garages

#check le type de bien
df = df[df["type_local"].isin(["Maison", "Appartement"])]

#sauvegareder la version propre
df.to_csv("dvf_clean.csv", index=False)
print("✅ Données nettoyées sauvegardées sous dvf_clean.csv")





✅ Données nettoyées sauvegardées sous dvf_clean.csv


In [10]:
import pandas as pd

df = pd.read_csv("dvf_clean.csv")
print("✅ Fichier chargé :", df.shape)
df.head(3)


✅ Fichier chargé : (65286, 10)


,date_mutation,valeur_fonciere,code_postal,nom_commune,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,prix_m2
0,2020-01-09,72000.0,1270.0,Coligny,Maison,35.0,2.0,5.350942,46.380918,2057.142857
1,2020-01-06,180300.0,1000.0,Bourg-en-Bresse,Maison,75.0,4.0,5.208899,46.196870,2404.000000
2,2020-01-06,54800.0,1000.0,Bourg-en-Bresse,Appartement,32.0,1.0,5.229862,46.206670,1712.500000
